# Results

## Get data

In [1]:
import time

import numpy as np
import pandas as pd
import plotly.express as px
import wandb
from transformers import AutoTokenizer

from helpers import get_data_from_file

template = "plotly_white"

fancy_cols = {'corr2incorr':            {"name": "Correct ⟶ Incorrect", "asc": True},
              'peak_ram_memory_mb':     {"name": "Peak RAM memory (MB)", "asc": True},
              'accuracy_sentences':     {"name": "Accuracy (sentences)", "asc": False},
              'gpu_memory_mb':          {"name": 'GPU memory (MB)', "asc": True},
              'incorr2incorr':          {"name": "Incorrect ⟶ Incorrect", "asc": True},
              'word_incorrection_rate': {"name": "Word incorrection rate", "asc": True},
              'ms_per_sentence':        {"name": "Inference time (ms/sentence)", "asc": True},
              'throughput_words':       {"name": "Throughput (words/s)", "asc": False},
              'accuracy_words':         {"name": "Accuracy (words)", "asc": False},
              'recall':                 {"name": "Recall", "asc": False},
              'incorr2corr':            {"name": "Incorrect ⟶ Correct", "asc": False},
              'corr2corr':              {"name": "Correct ⟶ Correct", "asc": False},
              'f05':                    {"name": "F0.5", "asc": False},
              'precision':              {"name": "Precision", "asc": False},
              'model_size':             {"name": "Model size (MB)", "asc": True},
              }

api = wandb.Api()
runs = api.runs("martin-elias-ctu-fit/Benchmarks")

run_dfs = []
for run in runs:
    run_df = run.history(keys=None)
    run_df["name"] = run.name
    # Rename old run metrics from token to word
    run_df.rename(columns={
            "throughput_tokens":       "throughput_words",
            "token_incorrection_rate": "word_incorrection_rate",
            "token_correction_rate":   "word_correction_rate",
            "accuracy_tokens":         "accuracy_words",

    }, inplace=True)
    run_dfs.append(run_df)
df = pd.concat(run_dfs, axis=0)

# Drop anything I don't care about in the graph
df.drop(columns=["_runtime", "_step", "_timestamp", "model_name", "skipped", 'ram_memory_mb', "should_skip",
                 "word_correction_rate"], inplace=True)

# Fix some values
df.loc[df.name == "jamspell", "model_size"] = 35
df.loc[df.name == "jamspell", "name"] = "JamSpell"
df.loc[df.name == "elmo-checker-finetuned", "name"] = "ELMO-SC-LSTM-Fine-tuned - space correction"
df.loc[df.name == "elmo-checker-pretrained", "name"] = "ELMO-SC-LSTM-Pre-trained - space correction"
df.loc[df.name == "elmo-checker-pretrained-wo_space_correction", "name"] = "ELMO-SC-LSTM-Pre-trained"
df.loc[df.name == "elmo-checker-finetuned-wo_space_correction", "name"] = "ELMO-SC-LSTM-Fine-tuned"

df.loc[df.name == "bert-checker-finetuned", "name"] = "BERT-Fine-tuned - space correction"
df.loc[df.name == "bert-checker-pretrained", "name"] = "BERT-Pre-trained - space correction"
df.loc[df.name == "bert-checker-pretrained-wo space correction", "name"] = "BERT-Pre-trained"
df.loc[df.name == "bert-checker-finetuned-wo space correction", "name"] = "BERT-Fine-tuned"
df.loc[df.name == "bert-checker-pretrained-wo space correction-tokenized", "name"] = "BERT-Pre-trained - token based"
df.loc[df.name == "bert-checker-finetuned-wo space correction-tokenized", "name"] = "BERT-Fine-tuned - token based"

data folder is set to `c:\users\brumda\documents\neuspell\neuspell\../neuspell_data` script


## BERT

In [7]:
bert_df = df[df["name"].str.contains("bert", case=False, na=False)].copy()
bert_df["name"] = bert_df["name"].str.replace("BERT-", "", case=False, regex=False).str.strip()

In [8]:
for col in bert_df:
    if col in ["accuracy_sentences", "accuracy_words", "word_incorrection_rate", "recall", "precision", "f05",
               "ms_per_sentence", ]:
        sorted_bert_df = bert_df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
        fig = px.bar(
                sorted_bert_df,
                x=col,
                y="name",
                color="name",
                title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
                labels={"name": "Version"},
                template=template,
        )
        fig.update_layout(title_x=0.5, xaxis_title='')
        # fig.show()


In [18]:
mx = "ms_per_sentence"
my = "accuracy_words"
# my = "f05"
title = "Accuracy"
# title = "F<sub>0.5</sub>"
size = 19

fig = px.scatter(
        bert_df,
        x=mx,
        y=my,
        color="name",
        title=f"NeuSpell BERT<br>{title} vs Speed",
        template=template,
)
fig.update_layout(
        title_x=0.5,
        yaxis=dict(range=[0, None]),
        xaxis_title="Inference Time (ms, lower is better)",
        # yaxis_title="F<sub>0.5</sub> (higher is better)",
        yaxis_title= "Accuracy words (higher is better)",
        xaxis=dict(autorange="reversed"),
        showlegend=False,
        font=dict(size=size),
)

fig.update_traces(marker={'size': 15})

fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Pre-trained - space correction", mx][0],
        y=bert_df.loc[bert_df.name == "Pre-trained - space correction", my][0],
        text="Pre-trained<br>space correction",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)
fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Fine-tuned - space correction", mx][0],
        y=bert_df.loc[bert_df.name == "Fine-tuned - space correction", my][0],
        text="Fine-tuned<br>space correction",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)
fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Pre-trained - token based", mx][0],
        y=bert_df.loc[bert_df.name == "Pre-trained - token based", my][0],
        text="Pre-trained<br>token based",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)
fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Fine-tuned - token based", mx][0],
        y=bert_df.loc[bert_df.name == "Fine-tuned - token based", my][0],
        text="Fine-tuned<br>token based",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)
fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Pre-trained", mx][0],
        y=bert_df.loc[bert_df.name == "Pre-trained", my][0],
        text="Pre-trained",
        showarrow=False,
        font=dict(size=size),
        xshift=-40, yshift=-20
)
fig.add_annotation(
        x=bert_df.loc[bert_df.name == "Fine-tuned", mx][0],
        y=bert_df.loc[bert_df.name == "Fine-tuned", my][0],
        text="Fine-tuned",
        showarrow=False,
        font=dict(size=size),
        xshift=40, yshift=20
)
fig.show()

In [10]:
fig.write_image(file="../thesis/images/bert_f05VSspeed.pdf", width=960, height=540, engine="kaleido")

## ELMO

In [11]:
elmo_df = df[df["name"].str.contains("elmo", case=False, na=False)].copy()
elmo_df["name"] = elmo_df["name"].str.replace("ELMO-SC-LSTM-", "", case=False, regex=False).str.strip()

In [12]:
mx = "ms_per_sentence"
my = "f05"
size = 19
fig = px.scatter(
        elmo_df,
        x=mx,
        y=my,
        color="name",
        # text='name',
        title="NeuSpell ELMO-SC-LSTM<br>F<sub>0.5</sub> vs Speed",
        # size="f05",
        # labels={"ms_per_sentence": "Inference Time (ms)", my: "Model Accuracy", "name": "Version"},
        template=template,
        # size_max=40,
)
fig.update_layout(
        title_x=0.5,
        yaxis=dict(range=[0, None]),
        xaxis_title="Inference Time (ms, lower is better)",
        yaxis_title="F<sub>0.5</sub> (higher is better)",
        xaxis=dict(autorange="reversed"),
        showlegend=False,
        font=dict(size=size),
)

fig.update_traces(marker={'size': 15})

fig.add_annotation(
        x=elmo_df.loc[elmo_df.name == "Pre-trained - space correction", mx][0],
        y=elmo_df.loc[elmo_df.name == "Pre-trained - space correction", my][0],
        text="Pre-trained<br>space correction",
        showarrow=False,
        font=dict(size=size),
        xshift=-10, yshift=-30
)
fig.add_annotation(
        x=elmo_df.loc[elmo_df.name == "Fine-tuned - space correction", mx][0],
        y=elmo_df.loc[elmo_df.name == "Fine-tuned - space correction", my][0],
        text="Fine-tuned<br>space correction",
        showarrow=False,
        font=dict(size=size),
        xshift=10, yshift=30
)
# fig.add_annotation(
#         x=elmo_df.loc[elmo_df.name == "Fine-tuned - token based", mx][0],
#         y=elmo_df.loc[elmo_df.name == "Fine-tuned - token based", my][0],
#         text="Fine-tuned<br>token based",
#         showarrow=False,
#         font=dict(size=size),
#         xshift=0, yshift=-30
# )
fig.add_annotation(
        x=elmo_df.loc[elmo_df.name == "Pre-trained", mx][0],
        y=elmo_df.loc[elmo_df.name == "Pre-trained", my][0],
        text="Pre-trained",
        showarrow=False,
        font=dict(size=size),
        xshift=-0, yshift=-20
)
fig.add_annotation(
        x=elmo_df.loc[elmo_df.name == "Fine-tuned", mx][0],
        y=elmo_df.loc[elmo_df.name == "Fine-tuned", my][0],
        text="Fine-tuned",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-20
)
fig.show()

In [13]:
fig.write_image(file="../thesis/images/elmo_f05VSspeed.pdf", width=960, height=540, engine="kaleido")

In [30]:
for col in elmo_df:
    if col in ["accuracy_sentences", "accuracy_words", "word_incorrection_rate", "recall", "precision", "f05",
               "ms_per_sentence", ]:
        sorted_elmo_df = elmo_df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
        fig = px.bar(
                sorted_elmo_df,
                x=col,
                y="name",
                color="name",
                title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
                labels={"name": "Version"},
        )
        fig.update_layout(title_x=0.5, xaxis_title='')
        # fig.show()


## ALL graphs

In [ ]:
# Get all the graphs
for col in df:
    if col in ["name", "inference_time", "throughput_sentences", 'typo_detection_model_inference_time',
               'typo_detection_model_ms_per_sentence', ]:
        continue
    sorted_df = df.sort_values(by=col, ascending=fancy_cols[col]["asc"])
    fig = px.bar(
            sorted_df,
            x=col,
            y="name",
            color="name",
            title=f'{fancy_cols[col]["name"]}<br>({"Lower" if fancy_cols[col]["asc"] else "Higher"} is better)',
            labels={"name": "Model"},
    )
    fig.update_layout(title_x=0.5, xaxis_title='')
    # fig.show()

In [2]:
best_options = df[df['name'].isin([
        "JamSpell",
        "BERT-Fine-tuned - space correction",
        "ELMO-SC-LSTM-Fine-tuned - space correction"
])]

In [5]:
mx = "ms_per_sentence"
my = "f05"
# my = "accuracy_words"
size = 19
fig = px.scatter(
        best_options,
        x=mx,
        y=my,
        color="name",
        title="Best variants of models<br>F<sub>0.5</sub> vs Speed",
        # title="Best variants of models<br>Accuracy vs Speed",
        template=template,
)
fig.update_layout(
        title_x=0.5,
        yaxis=dict(range=[0, None]),
        xaxis_title="Inference Time (ms, lower is better)",
        yaxis_title="F<sub>0.5</sub> (higher is better)",
        # yaxis_title="Accuracy words (higher is better)",
        xaxis=dict(autorange="reversed"),
        showlegend=False,
        font=dict(size=size),
)

fig.update_traces(marker={'size': 15})

fig.add_annotation(
        x=best_options.loc[best_options.name == "JamSpell", mx][0],
        y=best_options.loc[best_options.name == "JamSpell", my][0],
        text="JamSpell",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)

fig.add_annotation(
        x=best_options.loc[best_options.name == "BERT-Fine-tuned - space correction", mx][0],
        y=best_options.loc[best_options.name == "BERT-Fine-tuned - space correction", my][0],
        text="BERT",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)

fig.add_annotation(
        x=best_options.loc[best_options.name == "ELMO-SC-LSTM-Fine-tuned - space correction", mx][0],
        y=best_options.loc[best_options.name == "ELMO-SC-LSTM-Fine-tuned - space correction", my][0],
        text="ELMO-SC-LSTM",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)
fig.show()

In [6]:
fig.write_image(file="../thesis/images/models_comparison_f05VSspeed.pdf", width=960, height=540, engine="kaleido")
# fig.write_image(file="../thesis/images/models_comparison_accuracyVSspeed.pdf", width=960, height=540, engine="kaleido")

In [16]:
mx = "word_incorrection_rate"
my = "recall"
# my = "accuracy_words"
size = 19
fig = px.scatter(
        best_options,
        x=mx,
        y=my,
        color="name",
        title="Best variants of models<br>Recall vs Inccorection rate",
        # title="Best variants of models<br>Accuracy vs Speed",
        template=template,
)
fig.update_layout(
        title_x=0.5,
        yaxis=dict(range=[0, None]),
        xaxis_title="Incorrection rate (lower is better)",
        yaxis_title="Recall (higher is better)",
        # yaxis_title="Accuracy words (higher is better)",
        xaxis=dict(autorange="reversed"),
        showlegend=False,
        font=dict(size=size),
)

fig.update_traces(marker={'size': 15})

fig.add_annotation(
        x=best_options.loc[best_options.name == "JamSpell", mx][0],
        y=best_options.loc[best_options.name == "JamSpell", my][0],
        text="JamSpell",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)

fig.add_annotation(
        x=best_options.loc[best_options.name == "BERT-Fine-tuned - space correction", mx][0],
        y=best_options.loc[best_options.name == "BERT-Fine-tuned - space correction", my][0],
        text="BERT",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)

fig.add_annotation(
        x=best_options.loc[best_options.name == "ELMO-SC-LSTM-Fine-tuned - space correction", mx][0],
        y=best_options.loc[best_options.name == "ELMO-SC-LSTM-Fine-tuned - space correction", my][0],
        text="ELMO-SC-LSTM",
        showarrow=False,
        font=dict(size=size),
        xshift=0, yshift=-30
)
fig.show()

In [17]:
fig.write_image(file="../thesis/images/models_comparison_recallVSincorr_rate.pdf", width=960, height=540, engine="kaleido")

## Detect typo tokenizer max len

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("nreimers/MiniLM-L6-H384-uncased")

allDF = []
for name in ['train', 'dev', 'test']:
    df, _ = get_data_from_file(name)
    allDF.extend(df)

print(len(allDF))

token_lengths = []
start = time.time()
for text in allDF:
    tokens = tokenizer.encode(text)
    token_lengths.append(len(tokens))
print(f"Time taken to tokenize: {time.time() - start:.2f} seconds")

df_tokens = pd.DataFrame({'Token Length': token_lengths})

In [ ]:
max_len = 96
fig = px.histogram(
        df_tokens,
        x="Token Length",
        nbins=50,
        title="Distribution of Token Lengths in Dataset",
        labels={"Token Length": "Number of Tokens"},
        template=template,
)
fig.add_vline(
        x=max_len,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Current max_len: {max_len}",
        annotation_position="top right"
)
fig.update_layout(
        xaxis_title="Number of Tokens",
        yaxis_title="Number of Sentences",
        bargap=0.1
)
fig.show()

# Statistics
print(f"Maximum token length: {max(token_lengths)}")
print(f"Mean token length: {np.mean(token_lengths):.2f}")
print(f"Median token length: {np.median(token_lengths)}")
print(f"95th percentile: {np.percentile(token_lengths, 95)}")
print(f"99th percentile: {np.percentile(token_lengths, 99)}")
print(f"Percentage of sentences truncated: {sum(l > max_len for l in token_lengths) / len(token_lengths):.2%}")
